# Endangered Globe — Data Pipeline

Produces `animals.geojson`: one Point per threatened species, with label, IUCN category, and Wikipedia popularity.

**Steps**
1. Query IUCN Red List API → species list (EW / CR / EN / VU / NT + CD)
2. Clean & compute centroids from habitat polygons
3. Wikidata SPARQL → Wikipedia article title
4. Wikimedia Pageviews API → 12-month view count
5. Assemble & export GeoJSON

In [ ]:
# ── Dependencies ──────────────────────────────────────────────────────────────
# pip install requests geopandas shapely pandas tqdm

import os, time, json
import requests
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point, mapping
from tqdm.notebook import tqdm

---
## 0 · Configuration

In [ ]:
IUCN_TOKEN = "YOUR_TOKEN_HERE"          # https://api.iucnredlist.org/users/sign_up
USER_AGENT = "EndangeredGlobe/1.0 (t.boulademareuil@criteo.com)"  # Wikimedia requires this

TARGET_CATEGORIES = ["EW", "CR", "EN", "VU", "NT", "CD"]  # CD → displayed as NT
SLEEP_WIKI = 0.15   # seconds between Wikimedia requests
SLEEP_IUCN = 0.05

OUTPUT_PATH = "animals.geojson"

---
## 1 · IUCN Red List API — species list

The IUCN REST API v4 returns paginated pages of species. We filter by kingdom (Animalia) and iterate through all pages for each target category.

In [ ]:
IUCN_BASE = "https://apiv3.iucnredlist.org/api/v3"

def iucn_get(path, params=None):
    params = params or {}
    params["token"] = IUCN_TOKEN
    r = requests.get(f"{IUCN_BASE}{path}", params=params, timeout=30)
    r.raise_for_status()
    return r.json()

def fetch_species_by_category(category):
    """Return list of dicts: {taxonid, scientific_name, main_common_name, category}"""
    results, page = [], 0
    while True:
        data = iucn_get(f"/species/category/{category}", {"page": page})
        batch = data.get("result", [])
        if not batch:
            break
        results.extend(batch)
        page += 1
        time.sleep(SLEEP_IUCN)
    return results

# Fetch all categories
raw_species = []
for cat in TARGET_CATEGORIES:
    print(f"Fetching {cat}...", end=" ")
    batch = fetch_species_by_category(cat)
    print(f"{len(batch)} species")
    raw_species.extend(batch)

df = pd.DataFrame(raw_species)
print(f"\nTotal before filtering: {len(df):,}")
df.head(3)

In [ ]:
# Keep only Animalia, deduplicate (a species may appear in multiple pages)
# The API returns kingdom in the species detail endpoint — we'll filter there.
# Quick check on available columns:
print(df.columns.tolist())
df.category.value_counts()

In [ ]:
# Normalise: map CD → NT for display, deduplicate on taxonid
df["category_iucn"] = df["category"].replace("CD", "NT")
df = df.drop_duplicates(subset="taxonid")
print(f"After dedup: {len(df):,}")
df.category_iucn.value_counts()

---
## 2 · IUCN Spatial Data — habitat polygons → centroids

Two options:
- **Option A (recommended)**: download the IUCN bulk shapefiles from the Red List website (requires account). Load locally with GeoPandas.
- **Option B (API)**: call `/species/spatial/{taxonid}` per species — slow, rate-limited, avoid for large batches.

We use **Option A** here. Download the Animals shapefile from [IUCN Spatial Data Download](https://www.iucnredlist.org/resources/spatial-data-download) into the `data/` folder.

In [ ]:
# ── Option A: local shapefiles ────────────────────────────────────────────────
# Adjust this path to wherever you saved the IUCN shapefiles.
# The main file is typically named MAMMALS.shp, REPTILES.shp, etc.
# You can also use the combined "MULTIPLESPECIES" file if available.

SHP_PATHS = [
    "data/MAMMALS.shp",
    "data/AMPHIBIANS.shp",
    "data/REPTILES.shp",
    "data/BIRDS.shp",
    "data/FRESHWATER_FISHES.shp",
    # add others as needed
]

target_ids = set(df["taxonid"].astype(int))

gdfs = []
for path in SHP_PATHS:
    if not os.path.exists(path):
        print(f"Skipping (not found): {path}")
        continue
    print(f"Loading {path}...")
    g = gpd.read_file(path)
    # IUCN shapefiles use id_no or ID_NO for the species ID
    id_col = "id_no" if "id_no" in g.columns else "ID_NO"
    g = g[g[id_col].astype(int).isin(target_ids)]
    gdfs.append(g[[id_col, "geometry"]].rename(columns={id_col: "taxonid"}))

gdf_all = pd.concat(gdfs, ignore_index=True) if gdfs else gpd.GeoDataFrame()
print(f"Total polygons matched: {len(gdf_all):,}")

In [ ]:
# Compute centroid per species (union all polygons first, then centroid)
# This gives one point even when a species has multiple range polygons.
gdf_all["taxonid"] = gdf_all["taxonid"].astype(int)

centroids = (
    gdf_all
    .dissolve(by="taxonid")
    .reset_index()[["taxonid", "geometry"]]
)
centroids["geometry"] = centroids["geometry"].centroid
centroids["lon"] = centroids["geometry"].x
centroids["lat"] = centroids["geometry"].y

print(f"Centroids computed: {len(centroids):,}")

# Merge back into main df
df["taxonid"] = df["taxonid"].astype(int)
df = df.merge(centroids[["taxonid", "lon", "lat"]], on="taxonid", how="inner")
print(f"Species with spatial data: {len(df):,}")

---
## 3 · Wikidata SPARQL — IUCN ID → Wikipedia article title

Wikidata links IUCN species IDs to Wikipedia pages. One SPARQL query retrieves the mapping for all our IDs at once.

In [ ]:
WIKIDATA_ENDPOINT = "https://query.wikidata.org/sparql"

def build_sparql_query(iucn_ids):
    """Batch SPARQL: resolve IUCN taxon IDs to English Wikipedia article titles."""
    values = " ".join(f'"{i}"' for i in iucn_ids)
    return f"""
SELECT ?iucn_id ?article_title WHERE {{
  VALUES ?iucn_id {{ {values} }}
  ?taxon wdt:P627 ?iucn_id .          # P627 = IUCN taxon ID
  ?article schema:about ?taxon ;
            schema:inLanguage "en" ;
            schema:isPartOf <https://en.wikipedia.org/> .
  BIND(REPLACE(STR(?article), "https://en.wikipedia.org/wiki/", "") AS ?article_title)
}}
"""

def query_wikidata_batch(iucn_ids, batch_size=500):
    """Run SPARQL in batches to avoid query size limits."""
    mapping = {}
    ids = list(map(str, iucn_ids))
    for i in tqdm(range(0, len(ids), batch_size), desc="Wikidata batches"):
        batch = ids[i:i+batch_size]
        sparql = build_sparql_query(batch)
        r = requests.get(
            WIKIDATA_ENDPOINT,
            params={"query": sparql, "format": "json"},
            headers={"User-Agent": USER_AGENT},
            timeout=60
        )
        r.raise_for_status()
        for row in r.json()["results"]["bindings"]:
            iid = row["iucn_id"]["value"]
            title = row["article_title"]["value"]
            mapping[iid] = title
        time.sleep(1.0)  # Wikidata rate limit: be gentle
    return mapping

wikidata_map = query_wikidata_batch(df["taxonid"].tolist())
print(f"Wikipedia articles found: {len(wikidata_map):,} / {len(df):,}")

In [ ]:
# Attach wiki title; drop species with no Wikipedia article (no popularity signal)
df["wiki_title"] = df["taxonid"].astype(str).map(wikidata_map)
df_wiki = df.dropna(subset=["wiki_title"]).copy()
print(f"Remaining after wiki filter: {len(df_wiki):,}")

---
## 4 · Wikimedia Pageviews API — 12-month view count

Endpoint: `https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article/{project}/{access}/{agent}/{article}/monthly/{start}/{end}`

We use the last 12 completed months and sum the monthly totals.

In [ ]:
from datetime import date

# Last 12 completed months
today = date.today()
end_month   = date(today.year, today.month, 1)  # first of current month
start_month = date(end_month.year - 1, end_month.month, 1)
START = start_month.strftime("%Y%m01")
END   = (date(end_month.year, end_month.month, 1) - pd.DateOffset(months=1)).strftime("%Y%m01")
# Use simpler string math to avoid pandas offset in pure Python
end_month_prev = end_month.replace(day=1)
if end_month_prev.month == 1:
    end_month_prev = end_month_prev.replace(year=end_month_prev.year-1, month=12)
else:
    end_month_prev = end_month_prev.replace(month=end_month_prev.month-1)
END = end_month_prev.strftime("%Y%m01")

print(f"Pageview window: {START} → {END}")

PAGEVIEWS_BASE = "https://wikimedia.org/api/rest_v1/metrics/pageviews/per-article"

def get_pageviews(title):
    """Return total English Wikipedia views over the last 12 months, or 0 on error."""
    import urllib.parse
    encoded = urllib.parse.quote(title, safe="")
    url = f"{PAGEVIEWS_BASE}/en.wikipedia/all-access/all-agents/{encoded}/monthly/{START}/{END}"
    try:
        r = requests.get(url, headers={"User-Agent": USER_AGENT}, timeout=15)
        if r.status_code == 404:
            return 0
        r.raise_for_status()
        items = r.json().get("items", [])
        return sum(item["views"] for item in items)
    except Exception:
        return 0

# Run — ~0.15s per species; for 5k species: ~12 min
views = []
for title in tqdm(df_wiki["wiki_title"], desc="Pageviews"):
    views.append(get_pageviews(title))
    time.sleep(SLEEP_WIKI)

df_wiki["popularity"] = views
print(f"Median views: {int(df_wiki.popularity.median()):,}")
print(f"Zero-views: {(df_wiki.popularity == 0).sum()} species")
df_wiki.popularity.describe()

---
## 5 · Label selection & GeoJSON export

In [ ]:
# Label: prefer main_common_name (English common name), fall back to scientific_name
df_wiki["label"] = df_wiki["main_common_name"].where(
    df_wiki["main_common_name"].notna() & (df_wiki["main_common_name"] != ""),
    df_wiki["scientific_name"]
)

# Title-case common names
df_wiki["label"] = df_wiki["label"].str.title()

# Optional: drop species with 0 views (no Wikipedia presence → won't render usefully)
# df_wiki = df_wiki[df_wiki.popularity > 0]

print(f"Final species count: {len(df_wiki):,}")
df_wiki[["label", "category_iucn", "popularity", "lon", "lat"]].sample(10)

In [ ]:
# Build GeoJSON
features = []
for _, row in df_wiki.iterrows():
    features.append({
        "type": "Feature",
        "geometry": {
            "type": "Point",
            "coordinates": [round(row.lon, 4), round(row.lat, 4)]
        },
        "properties": {
            "label":         row.label,
            "category_iucn": row.category_iucn,
            "popularity":    int(row.popularity)
        }
    })

geojson = {"type": "FeatureCollection", "features": features}

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(geojson, f, ensure_ascii=False)

size_mb = os.path.getsize(OUTPUT_PATH) / 1e6
print(f"Written: {OUTPUT_PATH} — {len(features):,} features, {size_mb:.1f} MB")

---
## Quick sanity check

In [ ]:
# Load back and inspect
with open(OUTPUT_PATH) as f:
    check = json.load(f)

sample = check["features"][:5]
for feat in sample:
    p = feat["properties"]
    c = feat["geometry"]["coordinates"]
    print(f"{p['label']:40s} [{p['category_iucn']}]  {p['popularity']:>10,} views  ({c[1]:.2f}, {c[0]:.2f})")